imports part

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer

import warnings
warnings.filterwarnings("ignore")

download nltk resources

In [ ]:
nltk.download("vader_lexicon")
nltk.download("stopwords")

load daraset dataset info and missing values

In [ ]:
df = pd.read_csv("../data/bank_reviews_clean.csv")

df.head()
df.info()
df.isnull().sum()

Initialize Sentiment Analyzer

In [ ]:
sia = SentimentIntensityAnalyzer()

generate sentiment scores and create sentiment scores

In [ ]:
df["sentiment_score"] = df["review"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)


def classify_sentiment(score):

    if score > 0.05:
        return "Positive"

    elif score < -0.05:
        return "Negative"

    else:
        return "Neutral"

sentiment distribution

In [ ]:
df["sentiment_label"] = df["sentiment_score"].apply(
    classify_sentiment
)

df.head()
df["sentiment_label"].value_counts()

visualization: Sentiment by Bank 

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="bank",
    hue="sentiment_label"
)

plt.title("Sentiment Distribution by Bank")

plt.show()

 avarage sentiment by bank and rating vs sentiment

In [ ]:
df.groupby("bank")["sentiment_score"].mean()
df.groupby("rating")["sentiment_score"].mean()

text cleaning

In [ ]:
stop_words = stopwords.words("english")

tf-idf analysis

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20
)

X = tfidf.fit_transform(df["review"])

keywords = tfidf.get_feature_names_out()

keywords

keyword scores and visualization

In [ ]:
scores = X.sum(axis=0).A1

keyword_df = pd.DataFrame({
    "keyword": keywords,
    "score": scores
})

keyword_df = keyword_df.sort_values(
    by="score",
    ascending=False
)

keyword_df
plt.figure(figsize=(12, 6))

sns.barplot(
    data=keyword_df,
    x="score",
    y="keyword"
)

plt.title("Top Keywords in Reviews")

plt.show()

word cloud

In [ ]:
text = " ".join(df["review"].astype(str))

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(text)

plt.figure(figsize=(15, 7))

plt.imshow(wordcloud)

plt.axis("off")

plt.title("Word Cloud of Reviews")

plt.show()

Theme Identification

In [ ]:
def identify_theme(review):

    review = str(review).lower()

    if "login" in review or "password" in review:
        return "Account Access Issues"

    elif "slow" in review or "transfer" in review:
        return "Transaction Performance"

    elif "ui" in review or "design" in review:
        return "UI & Design"

    elif "otp" in review:
        return "OTP Issues"

    else:
        return "General Feedback"

Apply Themes and Theme Counts

In [ ]:
df["identified_theme"] = df["review"].apply(
    identify_theme
)

df.head()
df["identified_theme"].value_counts()

theme visualization

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    y="identified_theme",
    order=df["identified_theme"].value_counts().index
)

plt.title("Review Themes")

plt.show()

save processed data set

In [ ]:
df.to_csv(
    "../data/sentiment_analysis_results.csv",
    index=False
)

print("Processed dataset saved.")